# Modular Addition — Rebuttal Ablations

Compact 2×2 component ablations, train–test onset gaps, 100% follow-up, diagnostics, early stopping, and interruption-safe resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

P, FRAC_TRAIN = 113, 0.30
D_MODEL, NUM_HEADS, MLP_DIM = 128, 4, 512
LRS, SEEDS = [1e-4, 6e-4], list(range(1, 11))
LOG_EVERY, RECOVERY_EVERY = 50, 500
POST_99_GRACE, STABLE_100_LOGS = 1000, 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_ROOT = '/content/drive/MyDrive/grokking_rebuttal_addition'
os.makedirs(SAVE_ROOT, exist_ok=True)

RUN_COMPONENT_ABLATIONS = True
RUN_DIAGNOSTICS = True

CONFIGS = [
    # ---------------------------------------------------------
    # WD = 0
    # ---------------------------------------------------------
    dict(
        name="diag_sphere_bounded_lr_6e-4_wd0_long",
        model_type="spherical",
        bounded_head=True,
        tau=10.0,
        weight_decay=0.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),
    dict(
        name="diag_sphere_raw_lr_6e-4_wd0_long",
        model_type="spherical",
        bounded_head=False,
        tau=None,
        weight_decay=0.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),

    # ---------------------------------------------------------
    # WD = 1
    # ---------------------------------------------------------
    # Bu koşul zaten mevcut. Aynı isim sayesinde log + checkpoint
    # varsa otomatik olarak skip edilip summary'ye alınacak.
    dict(
        name="diag_sphere_bounded_lr_6e-4",
        model_type="spherical",
        bounded_head=True,
        tau=10.0,
        weight_decay=1.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),
    dict(
        name="diag_sphere_raw_lr_6e-4_wd1_long",
        model_type="spherical",
        bounded_head=False,
        tau=None,
        weight_decay=1.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),
]


def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def make_dataset(p, frac_train, seed):
    set_seed(seed)
    pairs = [(a, b) for a in range(p) for b in range(p)]
    random.Random(seed).shuffle(pairs)
    n_train = int(len(pairs) * frac_train)
    def build(part):
        x = torch.tensor([[a, b, p] for a, b in part], dtype=torch.long)
        y = torch.tensor([(a + b) % p for a, b in part], dtype=torch.long)
        return x, y
    train_x, train_y = build(pairs[:n_train])
    test_x, test_y = build(pairs[n_train:])
    return train_x, train_y, test_x, test_y


class AdditionTransformer(nn.Module):
    def __init__(self, p, model_type, bounded_head, tau):
        super().__init__()
        self.model_type, self.bounded_head, self.tau = model_type, bounded_head, tau
        self.d_head = D_MODEL // NUM_HEADS
        self.tok_embed = nn.Embedding(p + 1, D_MODEL)
        self.pos_embed = nn.Embedding(3, D_MODEL)
        self.W_Q = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_K = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_V = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_O = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.mlp_in = nn.Linear(D_MODEL, MLP_DIM)
        self.mlp_out = nn.Linear(MLP_DIM, D_MODEL)
        self.unembed = nn.Linear(D_MODEL, p, bias=False)
        if model_type == 'layernorm':
            self.input_norm = nn.LayerNorm(D_MODEL)
            self.norm1 = nn.LayerNorm(D_MODEL)
            self.norm2 = nn.LayerNorm(D_MODEL)

    def norm(self, h, which):
        if self.model_type == 'spherical':
            return F.normalize(h, dim=-1)
        return self.input_norm(h) if which == 0 else (self.norm1(h) if which == 1 else self.norm2(h))

    def forward(self, x, return_diag=False):
        _, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h_embed_raw = self.tok_embed(x) + self.pos_embed(pos)
        h0 = self.norm(h_embed_raw, 0)

        Q = self.W_Q(h0).view(-1, L, NUM_HEADS, self.d_head).transpose(1, 2)
        K = self.W_K(h0).view(-1, L, NUM_HEADS, self.d_head).transpose(1, 2)
        V = self.W_V(h0).view(-1, L, NUM_HEADS, self.d_head).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)
        attn_out = self.W_O((attn @ V).transpose(1, 2).contiguous().view(-1, L, D_MODEL))

        attn_pre = h0 + attn_out
        h1 = self.norm(attn_pre, 1)
        mlp_pre = h1 + self.mlp_out(F.relu(self.mlp_in(h1)))
        h2 = self.norm(mlp_pre, 2)
        h_final = h2[:, 2, :]

        if self.bounded_head:
            logits = F.linear(h_final, F.normalize(self.unembed.weight, dim=1)) * float(self.tau)
        else:
            logits = self.unembed(h_final)

        if not return_diag:
            return logits

        eps = 1e-12
        uniform = torch.full_like(attn, 1.0 / L)
        entropy = -(attn.clamp_min(eps) * attn.clamp_min(eps).log()).sum(dim=-1)
        return logits, {
            'embed_raw_norm': h_embed_raw.norm(dim=-1).mean().item(),
            'input_post_norm': h0.norm(dim=-1).mean().item(),
            'attn_residual_pre_norm': attn_pre.norm(dim=-1).mean().item(),
            'attn_residual_post_norm': h1.norm(dim=-1).mean().item(),
            'mlp_residual_pre_norm': mlp_pre.norm(dim=-1).mean().item(),
            'mlp_residual_post_norm': h2.norm(dim=-1).mean().item(),
            'final_hidden_norm': h_final.norm(dim=-1).mean().item(),
            'logit_l2_norm': logits.norm(dim=-1).mean().item(),
            'max_abs_logit': logits.abs().amax(dim=-1).mean().item(),
            'attention_entropy': entropy.mean().item(),
            'attention_distance_to_uniform': (attn - uniform).pow(2).sum(dim=-1).sqrt().mean().item(),
            'unembed_row_norm_mean': self.unembed.weight.norm(dim=1).mean().item(),
            'unembed_row_norm_std': self.unembed.weight.norm(dim=1).std().item(),
        }


def parameter_norms(model):
    groups = {k: [] for k in ['global', 'embeddings', 'attention', 'mlp', 'unembedding']}
    for name, p in model.named_parameters():
        sq = p.detach().float().pow(2).sum()
        groups['global'].append(sq)
        if 'embed' in name: groups['embeddings'].append(sq)
        elif any(k in name for k in ['W_Q', 'W_K', 'W_V', 'W_O']): groups['attention'].append(sq)
        elif 'mlp_' in name: groups['mlp'].append(sq)
        elif 'unembed' in name: groups['unembedding'].append(sq)
    return {f'param_norm_{k}': math.sqrt(torch.stack(v).sum().item()) if v else 0.0 for k, v in groups.items()}


def first_onset(history, key, threshold):
    for epoch, value in zip(history['epochs'], history[key]):
        if value >= threshold:
            return epoch
    return None


def train_one(model, train_x, train_y, test_x, test_y, cfg, recovery_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'], betas=(0.9, 0.999))
    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)
    history = {k: [] for k in ['epochs', 'train_acc', 'test_acc', 'train_loss', 'test_loss']}
    diagnostics, start_epoch, test99_epoch, stable100 = [], 0, None, 0
    stop_reason = 'max_epochs'

    if os.path.exists(recovery_path):
        state = torch.load(recovery_path, map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        history, diagnostics = state['history'], state['diagnostics']
        start_epoch = state['epoch'] + 1
        test99_epoch, stable100 = state['test99_epoch'], state['stable100']
        print('resume:', cfg['name'], 'from', start_epoch)

    for epoch in tqdm(range(start_epoch, cfg['max_epochs']), desc=cfg['name'], leave=False):
        model.train()
        loss = criterion(model(train_x), train_y)
        if not torch.isfinite(loss):
            stop_reason = f'non_finite_loss_epoch_{epoch}'
            break
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()

        if epoch % LOG_EVERY != 0 and epoch != cfg['max_epochs'] - 1:
            continue

        model.eval()
        with torch.no_grad():
            train_logits = model(train_x)
            if cfg['instrument']:
                test_logits, diag = model(test_x, return_diag=True)
                diag.update(parameter_norms(model)); diag['epoch'] = epoch
                diagnostics.append(diag)
            else:
                test_logits = model(test_x)
            train_loss = criterion(train_logits, train_y).item()
            test_loss = criterion(test_logits, test_y).item()
            train_acc = (train_logits.argmax(-1) == train_y).float().mean().item()
            test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

        for k, v in [('epochs', epoch), ('train_acc', train_acc), ('test_acc', test_acc),
                     ('train_loss', train_loss), ('test_loss', test_loss)]:
            history[k].append(v)

        if test99_epoch is None and test_acc >= 0.99:
            test99_epoch = epoch
        stable100 = stable100 + 1 if test_acc >= 1.0 - 1e-12 else 0

        if epoch % RECOVERY_EVERY == 0:
            torch.save(dict(epoch=epoch, model_state_dict=model.state_dict(),
                            optimizer_state_dict=optimizer.state_dict(), history=history,
                            diagnostics=diagnostics, test99_epoch=test99_epoch,
                            stable100=stable100), recovery_path)

        if stable100 >= STABLE_100_LOGS:
            stop_reason = 'stable_test_100'
            break
        if test99_epoch is not None and epoch >= test99_epoch + POST_99_GRACE:
            stop_reason = 'post_99_grace_complete'
            break

    return history, diagnostics, stop_reason


def make_summary(cfg, seed, history, stop_reason):
    out = dict(config=cfg['name'], seed=seed, model_type=cfg['model_type'],
               bounded_head=cfg['bounded_head'], tau=cfg['tau'],
               weight_decay=cfg['weight_decay'], lr=cfg['lr'],
               max_epochs=cfg['max_epochs'], stop_reason=stop_reason,
               last_epoch=history['epochs'][-1] if history['epochs'] else None,
               peak_test_acc=max(history['test_acc']) if history['test_acc'] else None)
    for label, threshold in [('95', 0.95), ('99', 0.99), ('100', 1.0 - 1e-12)]:
        tr = first_onset(history, 'train_acc', threshold)
        te = first_onset(history, 'test_acc', threshold)
        out[f'train_epoch_{label}'] = tr
        out[f'test_epoch_{label}'] = te
        out[f'gap_{label}'] = None if tr is None or te is None else te - tr
    return out


rows = []
for cfg in CONFIGS:
    cfg_dir = os.path.join(SAVE_ROOT, cfg['name'])
    os.makedirs(cfg_dir, exist_ok=True)
    for seed in cfg['seeds']:
        stem = f"{cfg['name']}_seed{seed}"
        log_path = os.path.join(cfg_dir, stem + '.json')
        ckpt_path = os.path.join(cfg_dir, stem + '.pt')
        recovery_path = os.path.join(cfg_dir, stem + '_recovery.pt')

        if os.path.exists(log_path) and os.path.exists(ckpt_path):
            with open(log_path) as f: saved = json.load(f)
            rows.append(saved['summary']); print('skip:', stem); continue

        train_x, train_y, test_x, test_y = make_dataset(P, FRAC_TRAIN, seed)
        set_seed(seed)
        model = AdditionTransformer(P, cfg['model_type'], cfg['bounded_head'], cfg['tau']).to(DEVICE)
        history, diagnostics, stop_reason = train_one(
            model, train_x, train_y, test_x, test_y, cfg, recovery_path
        )
        summary = make_summary(cfg, seed, history, stop_reason)

        torch.save(dict(model_state_dict=model.state_dict(), summary=summary, config=cfg), ckpt_path)
        with open(log_path, 'w') as f:
            json.dump(dict(summary=summary, history=history, diagnostics=diagnostics), f, indent=2)
        if os.path.exists(recovery_path): os.remove(recovery_path)

        rows.append(summary)
        pd.DataFrame(rows).to_csv(os.path.join(SAVE_ROOT, 'summary_incremental.csv'), index=False)
        print('saved:', stem)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()


df = pd.DataFrame(rows)
df.to_csv(os.path.join(SAVE_ROOT, 'summary_final.csv'), index=False)
print('\nPer-run results:')
print(df.sort_values(['config', 'seed']).to_string(index=False))

agg = (df.groupby('config', dropna=False)
       .agg(seeds=('seed', 'count'),
            reached_95=('test_epoch_95', lambda x: x.notna().sum()),
            reached_99=('test_epoch_99', lambda x: x.notna().sum()),
            reached_100=('test_epoch_100', lambda x: x.notna().sum()),
            mean_test_95=('test_epoch_95', 'mean'), std_test_95=('test_epoch_95', 'std'),
            mean_test_99=('test_epoch_99', 'mean'), std_test_99=('test_epoch_99', 'std'),
            mean_gap_95=('gap_95', 'mean'), mean_gap_99=('gap_99', 'mean'),
            mean_peak_test_acc=('peak_test_acc', 'mean'))
       .reset_index())
agg.to_csv(os.path.join(SAVE_ROOT, 'configuration_summary.csv'), index=False)
print('\nConfiguration summary:')
print(agg.to_string(index=False))
